In [ ]:
!pip install -U "bitsandbytes>=0.44.1" \
             transformers==4.44.0 \
             trl==0.9.4 \
             peft==0.11.1 \
             accelerate==0.31.0 \
             datasets==2.20.0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 54.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 226.7/226.7 kB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.6/251.6 kB 28.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 309.4/309.4 kB 31.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 547.8/547.8 kB 46.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 45.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.1/316.1 kB 30.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 123.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.6/179.6 kB 11.9 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.0
    Uninstalling fsspec-2025.3.0:
      Successfully uninstalled fsspec-2025.3.0
  Attempting uninstall: tokenizers
    

In [ ]:
import torch
import gc
import json
import subprocess
import os
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel, LoraConfig
from trl import AutoModelForCausalLMWithValueHead, PPOConfig, PPOTrainer
from datasets import load_dataset
from tqdm import tqdm
from huggingface_hub import hf_hub_download

def extract_md_code_block(st):
    if "```" not in st:
        return st
    else:
        return st[st.index("```") + 3:].replace("```", "")

def can_compile_assembly(asm_code: str, arch: str = "x86_64", verbose: bool = False) -> bool:
    arch_flag = "-m64" if arch.lower() in ("x86_64", "amd64") else "-m32"
    devnull = "/dev/null" if os.name != "nt" else "NUL"

    cmd = [
        "gcc",
        arch_flag,
        "-x",
        "assembler",
        "-c",
        "-",
        "-o",
        devnull,
    ]

    try:
        result = subprocess.run(
            cmd,
            input=asm_code.encode("utf-8"),
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            check=False,
        )
    except FileNotFoundError:
        if verbose:
            print("Error: gcc not found in PATH")
        return False

    if result.returncode != 0:
        if verbose:
            print("Assembler diagnostics:")
            print(result.stderr.decode("utf-8", errors="replace"))
        return False

    return True

def longestCommonSubsequence(text1, text2):
    m, n = len(text1), len(text2)
    dp_row1 = [0] * (n + 1)
    dp_row2 = [0] * (n + 1)

    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if text1[i - 1] == text2[j - 1]:
                dp_row2[j] = 1 + dp_row1[j - 1]
            else:
                dp_row2[j] = max(dp_row2[j - 1], dp_row1[j])
        dp_row1 = dp_row2
        dp_row2 = [0] * (n + 1)

    return dp_row1[-1]

def levenshtein(a: str, b: str) -> int:
    if a == b:
        return 0
    la, lb = len(a), len(b)
    if la == 0:
        return lb
    if lb == 0:
        return la
    if la < lb:
        a, b = b, a
        la, lb = lb, la
    prev = list(range(lb + 1))
    for i, ca in enumerate(a, start=1):
        curr = [i]
        for j, cb in enumerate(b, start=1):
            ins = prev[j] + 1
            delete = curr[j - 1] + 1
            replace = prev[j - 1] + (0 if ca == cb else 1)
            curr.append(min(ins, delete, replace))
        prev = curr
    return prev[-1]

def compute_rewards(generated_texts, ground_truths):
    rewards = []
    for gen, ref in zip(generated_texts, ground_truths):
        gen_clean = extract_md_code_block(gen).strip()
        ref_clean = ref.strip()

        compiles = can_compile_assembly(gen_clean)
        compile_score = 1.0 if compiles else 0.0

        lcs_len = longestCommonSubsequence(gen_clean, ref_clean)
        lcs_score = lcs_len / max(len(ref_clean), 1)

        total = (0.5 * compile_score) + (0.5 * lcs_score)
        rewards.append(torch.tensor(total))
    return rewards

In [ ]:
gc.collect()
torch.cuda.empty_cache()

MODEL_ID = "codellama/CodeLlama-7b-Instruct-hf"
ADAPTER_ID = "SongTonyLi/codellama-4bit-peft-lora-r32-sft-c2x86"

config_path = hf_hub_download(repo_id=ADAPTER_ID, filename="adapter_config.json")
with open(config_path, "r") as f:
    raw_config = json.load(f)

valid_keys = {
    "r", "lora_alpha", "target_modules", "lora_dropout", "bias", "task_type",
    "modules_to_save", "layers_to_transform", "layers_pattern", "rank_pattern",
    "alpha_pattern", "init_lora_weights"
}
clean_config = {k: v for k, v in raw_config.items() if k in valid_keys}
peft_config = LoraConfig(**clean_config)

bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

base_inner = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb_config, device_map={"": 0})
base_inner = PeftModel.from_pretrained(base_inner, ADAPTER_ID, is_trainable=True, config=peft_config)
base_model = AutoModelForCausalLMWithValueHead(base_inner)
base_model.gradient_checkpointing_enable()
base_model.pretrained_model.enable_input_require_grads()
base_model.is_peft_model = True

ref_inner = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb_config, device_map={"": 0})
ref_inner = PeftModel.from_pretrained(ref_inner, ADAPTER_ID, is_trainable=False, config=peft_config)
ref_model = AutoModelForCausalLMWithValueHead(ref_inner)
ref_model.eval()

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

AutoModelForCausalLMWithValueHead(
  (pretrained_model): PeftModelForCausalLM(
    (base_model): LoraModel(
      (model): LlamaForCausalLM(
        (model): LlamaModel(
          (embed_tokens): Embedding(32016, 4096)
          (layers): ModuleList(
            (0-31): 32 x LlamaDecoderLayer(
              (self_attn): LlamaSdpaAttention(
                (q_proj): lora.Linear4bit(
                  (base_layer): Linear4bit(in_features=4096, out_features=4096, bias=False)
                  (lora_dropout): ModuleDict(
                    (default): Identity()
                  )
                  (lora_A): ModuleDict(
                    (default): Linear(in_features=4096, out_features=32, bias=False)
                  )
                  (lora_B): ModuleDict(
                    (default): Linear(in_features=32, out_features=4096, bias=False)
                  )
                  (lora_embedding_A): ParameterDict()
                  (lora_embedding_B): ParameterDict()
                )

In [ ]:
DATASET_ID = "AzizZeynalli/c2x86-dataset"
train_dataset = load_dataset(DATASET_ID, split="train").select(range(5000, 5300))
val_dataset = load_dataset(DATASET_ID, split="validation").shuffle(seed=42).select(range(200))

SYSTEM_PROMPT = "You are a compiler assistant who is able to compile C code into x86 assembly code in ATT syntax. You are working on Debian OS and you don’t need to generate non-essential directives. Please provide your generated code directly without any prefix."

In [ ]:
import torch
import gc
import bitsandbytes as bnb
from tqdm import tqdm

if 'ppo_trainer' in locals(): del ppo_trainer
if 'optimizer' in locals(): del optimizer
if 'query_tensors' in locals(): del query_tensors
if 'response_tensors' in locals(): del response_tensors
gc.collect()
torch.cuda.empty_cache()

config = PPOConfig(
    model_name="codellama_ppo_paged",
    learning_rate=1e-6,
    batch_size=16,
    mini_batch_size=1,
    gradient_accumulation_steps=16,
    optimize_cuda_cache=True,
    ppo_epochs=1,
    init_kl_coef=0.1,
    adap_kl_ctrl=True,
    target_kl=0.1,
    use_score_scaling=True,
    use_score_norm=True,
    cliprange=0.1,
)

optimizer = bnb.optim.PagedAdamW32bit(base_model.parameters(), lr=config.learning_rate)

ppo_trainer = PPOTrainer(config=config, model=base_model, ref_model=ref_model, tokenizer=tokenizer, optimizer=optimizer)

gen_kwargs = {
    "min_length": -1,
    "top_k": 0.0,
    "top_p": 0.9,
    "do_sample": True,
    "pad_token_id": tokenizer.eos_token_id,
    "max_new_tokens": 256
}

num_batches = len(train_dataset) // config.batch_size
pbar = tqdm(range(num_batches))

for i in pbar:
    torch.cuda.empty_cache()

    batch_data = train_dataset[i*config.batch_size : (i+1)*config.batch_size]

    formatted_prompts = []
    for c_code in batch_data['c']:
        messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": f"Please compile the following code \n {c_code}"}]
        formatted_prompts.append(tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))

    encodings = [tokenizer(p, return_tensors="pt") for p in formatted_prompts]
    query_tensors = [e["input_ids"][0].cuda() for e in encodings]
    query_masks = [e["attention_mask"][0].cuda() for e in encodings]

    response_tensors = []
    base_model.config.use_cache = True

    for q_tensor, q_mask in zip(query_tensors, query_masks):
        single_response = ppo_trainer.generate(
            q_tensor,
            return_prompt=False,
            attention_mask=q_mask.unsqueeze(0),
            **gen_kwargs
        )
        response_tensors.append(single_response.squeeze())

    base_model.config.use_cache = False

    gen_texts = [tokenizer.decode(r, skip_special_tokens=True) for r in response_tensors]
    rewards = compute_rewards(gen_texts, batch_data['s'])

    stats = ppo_trainer.step(query_tensors, response_tensors, rewards)

    reward_avg = torch.stack(rewards).mean().item()
    pbar.set_description(f"L: {stats['ppo/loss/total']:.4f} | R: {reward_avg:.3f}")

ppo_trainer.save_pretrained("codellama-c2x86-ppo-rl")

L: 0.0303 | R: 0.300:   6%|▌         | 1/18 [08:27<2:23:48, 507.53s/it]`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:85: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(
L: 0.0304 | R: 0.340:  11%|█         | 2/18 [19:03<2:35:26, 582.91s/it]/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:85: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(
L: 0.0312 | R: 0.295:  17%|█▋        | 3/18 [29:56<2:33:44, 614.94s/it]/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:85: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(
L: 0.0441 | R: 0.299:  22%|██▏       | 4/18 [40:06<2:23:00, 612.90s/it]/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:85: UserWarning: None of the inputs have requires_grad=True.

OutOfMemoryError: CUDA out of memory. Tried to allocate 318.00 MiB. GPU 0 has a total capacity of 39.56 GiB of which 896.00 KiB is free. Process 61337 has 39.42 GiB memory in use. Of the allocated memory 36.07 GiB is allocated by PyTorch, and 2.84 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
import torch
from huggingface_hub import HfApi, login

login()

REPO_ID = "AzizZeynalli/codellama-7b-c2x86-ppo-rl"
local_dir = "codellama-c2x86-ppo-rl"

os.makedirs(local_dir, exist_ok=True)

base_model.save_pretrained(local_dir)
tokenizer.save_pretrained(local_dir)

api = HfApi()
api.create_repo(repo_id=REPO_ID, exist_ok=True, repo_type="model")
api.upload_folder(folder_path=local_dir, repo_id=REPO_ID, repo_type="model")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...86-ppo-rl/tokenizer.model:  95%|#########4|  473kB /  500kB            

  ...adapter_model.safetensors:   0%|          |  558kB /  320MB            

  ...-ppo-rl/pytorch_model.bin:   1%|1         |   198B / 18.3kB            

CommitInfo(commit_url='https://huggingface.co/AzizZeynalli/codellama-7b-c2x86-ppo-rl/commit/5c3e491a7013708c555778ec66f1fb0149b6c5ea', commit_message='Upload folder using huggingface_hub', commit_description='', oid='5c3e491a7013708c555778ec66f1fb0149b6c5ea', pr_url=None, repo_url=RepoUrl('https://huggingface.co/AzizZeynalli/codellama-7b-c2x86-ppo-rl', endpoint='https://huggingface.co', repo_type='model', repo_id='AzizZeynalli/codellama-7b-c2x86-ppo-rl'), pr_revision=None, pr_num=None)

In [ ]:
import numpy as np
from tqdm import tqdm
import tempfile

ADAPTER_ID = "AzizZeynalli/codellama-7b-c2x86-ppo-rl"
BASE_MODEL = "codellama/CodeLlama-7b-Instruct-hf"
DATASET_ID = "AzizZeynalli/c2x86-dataset"
SYSTEM_PROMPT = "You are a compiler assistant who is able to compile C code into x86 assembly code in ATT syntax. You are working on Debian OS and you don’t need to generate non-essential directives. Please provide your generated code directly without any prefix."


tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto"
)

model = PeftModel.from_pretrained(base_model, ADAPTER_ID)
model.eval()

def extract_assembly_clean(text):
    if "[/INST]" in text:
        text = text.split("[/INST]")[-1]
    if "```" in text:
        try:
            return text.split("```")[1].strip()
        except:
            pass
    return text.replace("```", "").strip()

val_dataset = load_dataset(DATASET_ID, split="validation").shuffle(seed=42).select(range(50))
metrics = {"compile": [], "lcs": [], "levenshtein": []}
eval_kwargs = {"max_new_tokens": 512, "do_sample": False, "pad_token_id": tokenizer.eos_token_id}
BATCH_SIZE = 1

for i in tqdm(range(0, len(val_dataset), BATCH_SIZE)):
    batch = val_dataset.select(range(i, min(i + BATCH_SIZE, len(val_dataset))))

    prompts = []
    for c_code in batch['c']:
        messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": f"Please compile the following code \n {c_code}"}]
        prompts.append(tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))

    inputs = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True).to("cuda")

    with torch.no_grad():
        outputs = model.generate(**inputs, **eval_kwargs)

    generated_texts = tokenizer.batch_decode(outputs, skip_special_tokens=True)

    for gen_text, target_asm in zip(generated_texts, batch['s']):
        gen_clean = extract_assembly_clean(gen_text)
        ref_clean = target_asm.strip()
        metrics["compile"].append(1.0 if can_compile_assembly(gen_clean) else 0.0)
        lcs_len = longestCommonSubsequence(gen_clean, ref_clean)
        metrics["lcs"].append(lcs_len / max(len(ref_clean), 1))
        lev_dist = levenshtein(gen_clean, ref_clean)
        metrics["levenshtein"].append(lev_dist / max(len(ref_clean), 1))

print(f"Compile Rate:      {np.mean(metrics['compile']):.4f}")
print(f"Norm LCS:          {np.mean(metrics['lcs']):.4f}")
print(f"Norm Levenshtein:  {np.mean(metrics['levenshtein']):.4f}")

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

100%|██████████| 50/50 [47:29<00:00, 56.99s/it]

Compile Rate:      0.4600
Norm LCS:          0.4450
Norm Levenshtein:  0.5641


In [ ]:
val_ds = load_dataset(DATASET_ID, split="validation").shuffle(seed=42).select(range(10))
results = []
model.eval()

for row in tqdm(val_ds):
    c = row['c']
    tgt = row['s']

    msgs = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": f"Please compile the following code \n {c}"}]
    prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        out_ppo = model.generate(**inputs, max_new_tokens=512, do_sample=False)

        with model.disable_adapter():
            out_base = model.generate(**inputs, max_new_tokens=512, do_sample=False)

    pred_ppo = extract_assembly_clean(tokenizer.decode(out_ppo[0], skip_special_tokens=True))
    pred_base = extract_assembly_clean(tokenizer.decode(out_base[0], skip_special_tokens=True))

    results.append({
        "C": c,
        "Target": tgt,
        "Base": pred_base,
        "PPO": pred_ppo
    })

for i in range(min(3, len(results))):
    r = results[i]
    print(f"-EXAMPLE {i+1}")
    print(f"-INPUT C:\n{r['C'][:200]}...")
    print("=" * 50)
    print(f"-BASE MODEL:\n{r['Base'][:300]}...")
    print("=" * 50)
    print(f"PPO MODEL (RL):\n{r['PPO'][:300]}...")
    print("=" * 50)
    print(f"TARGET TRUTH:\n{r['Target'][:300]}...")

  0%|          | 0/10 [00:00<?, ?it/s]Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
 10%|█         | 1/10 [01:31<13:43, 91.55s/it]Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
 20%|██        | 2/10 [03:02<12:10, 91.26s/it]Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
 30%|███       | 3/10 [04:05<09:09, 78.46s/it]Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
 40%|████      | 4/10 [05:36<08:20, 83.36s/it]Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
 50%|█████     | 5/10 [06:41<06:22, 76.59s/it]Setting `pad_token_id` to `eos_token_id`:2 for open-

-EXAMPLE 1
-INPUT C:
#include <stdio.h>


long long gcd(long long u, long long v)
{
    long long shift;
    if (u == 0L) return v;
    if (v == 0L) return u;
    for (shift = 0L; ((u | v) & 1L) == 0L; ++shift)
    {
    ...
-BASE MODEL:
.file   "fraction.c"
    .text
    .globl  gcd
    .type   gcd, @function
gcd:
.LFB0:
    .cfi_startproc
    pushq   %rbp
    .cfi_def_cfa_offset 16
    .cfi_offset 6, -16
    movq    %rsp, %rbp
    .cfi_def_cfa_register 6
    movq    %rdi, -8(%rbp)
    movq    %rsi, -16(%rbp)
    movq    $0, -24(%r...
PPO MODEL (RL):
.text
	.globl	gcd
gcd:
	pushq	%rbp
	movq	%rsp, %rbp
	movq	%rdi, -24(%rbp)
	movq	%rsi, -32(%rbp)
	cmpq	$0, -24(%rbp)
	jne	.L2
	movq	-32(%rbp), %rax
	jmp	.L3
.L2:
	cmpq	$0, -32(%rbp)
	jne	.L4
	movq	-24(%rbp), %rax
	jmp	.L3
.L4:
	movq	$0, -8(%rbp)
	jmp	.L5
.L6:
	movq	-24(%rbp), %rax
	sarq	%rax
	movq	%r...
TARGET TRUTH:
	.text
	.globl	gcd
gcd:
	pushq	%rbp
	movq	%rsp, %rbp
	movq	%rdi, -24(%rbp)
	movq	%rsi, -32(%rbp)
	cmpq	$0, -24(%rbp)
	jne	.L